# Modelo SIR

Simulando uma epidemia usando Matemática e Computação

Breno Pinna  
João A. Proença  
27 de novembro de 2025

# Introduzindo o modelo SIR <!-- esses sao os slides de titulo principais -->

## Apresentação teórica <!-- esses sao os normais, com conteudo -->

Seja uma doença infecciosa, causada por algum agente biológico,
transmitida através do contato entre infectados e saudáveis.

Dividiremos a população em 3 grupos
($\textcolor{ForestGreen}{S}\textcolor{red}{I}\textcolor{blue}{R}$):

-   $\mathrm{\textcolor{ForestGreen}{Suscetíveis}}$ à contaminação:
    nunca apresentaram a doença, mas podem contrair;
-   $\mathrm{\textcolor{red}{Infectados}}$: atualmente infectados com a
    doença;
-   $\mathrm{\textcolor{blue}{Removidos}}$: foram infectados e ganharam
    imunidade permanente.

Suponhamos que não ocorrem nascimentos ou mortes durante o período
analisado, ou seja, o número de indivíduos
$N = \textcolor{ForestGreen}{S}(t) + \textcolor{red}{I}(t) + \textcolor{blue}{R}(t)$
é constante.

## Modelando matematicamente os grupos

O modelo
$\textcolor{ForestGreen}{S}\textcolor{red}{I}\textcolor{blue}{R}$ é
caracterizado pelas seguintes equações:

$$
\frac{dS}{dt} = - \beta S I \qquad (\beta = \tau \mu) \tag{1}
$$

$$
\frac{dI}{dt} = \beta S I - \gamma I \tag{2}
$$

$$
\frac{dR}{dt} = \gamma I \tag{3}
$$

-   $SI$: número de pares possíveis de um suscetível e um infectado;
-   $\mu$: probabilidade por unidade de tempo de ocorrer um encontro SI;
-   $\tau$: probabilidade de um encontro resultar em contágio;
-   $\beta$: taxa de transmissão da doença;
-   $\gamma$: taxa de recuperação dos infectados.

## Interpretando as equações

Supondo $S(0) \approx N$ e $0 < I(0) \ll N$ (condições iniciais, supondo
uma grande maioria suscetível), temos que a equação $(2)$ só vai gerar
uma epidemia caso $\frac{dI(0)}{dt} > 0$, ou seja:

$$
\beta S(0) I(0) - \gamma I(0) > 0
$$

$$
\beta S(0) I(0) > \gamma I(0) \qquad (\div \ \gamma I(0))
$$

$$
\frac{\beta}{\gamma} S(0) > 1 \tag{4}
$$

------------------------------------------------------------------------

Chamamos o termo $R_0 = (\beta / \gamma) S(0)$ de **número básico de
reprodução**, que indica quantos indivíduos suscetíveis serão infectados
por cada indivíduo infectado.

Quanto maior ele for, mais rápido uma doença vai se espalhar. Se
$R_0 < 1$, o número de infectados cai e não há epidemia.

Reescrevendo a equação $(2)$ com esse novo conceito, temos:

$$
\frac{dI}{dt}\Bigg|_{t=0} = \gamma\,(R_0 - 1)\,I(0) \tag{5}
$$

Com isso, vemos que o crescimento inicial do número de infectados é
diretamente proporcional ao valor de $R_0$, sendo esse parâmetro uma
forma de avaliar se ocorrerá ou não uma epidemia.

------------------------------------------------------------------------

Este sistema de equações diferenciais acopladas não possui solução
analítica conhecida. Portanto, a solução é obtida numericamente. Desta
vez, utilizaremos o método de Runge-Kutta de ordem 4 (RK4) para resolver
este sistema.

# O método de Runge-Kutta de ordem 4 (RK4)

## Descrevendo o funcionamento

O método RK4 resolve sistemas de equações diferenciais acopladas do tipo
$\frac{d\vec{r}}{dt} = \vec{f}(\vec{r}, t)$. Ele anda pequenos passos de
tamanho $h$ a cada ciclo, e calcula iterativamente as soluções, seguindo
a estrutura:

$$
\begin{aligned}
  \vec{k}_1 &= h \vec{f}(\vec{r}_n, t_n), \\
  \vec{k}_2 &= h \vec{f}\!\left(\vec{r}_n + \frac{1}{2}\vec{k}_1,\, t_n + \frac{1}{2}h\right), \\
  \vec{k}_3 &= h \vec{f}\!\left(\vec{r}_n + \frac{1}{2}\vec{k}_2,\, t_n + \frac{1}{2}h\right), \\
  \vec{k}_4 &= h \vec{f}\!\left(\vec{r}_n + \vec{k}_3,\, t_n + h\right), \\
  \vec{r}(t_n+h) &= \vec{r}_{n+1} = \vec{r}_n(t_n)
    + \frac{1}{6}\left(\vec{k}_1 + 2\vec{k}_2 + 2\vec{k}_3 + \vec{k}_4 \right), \\
\end{aligned}
$$

onde $\vec{r}$ é o vetor cujos elementos são as funções que estão sendo
derivadas e $\vec{f}(\vec{r}, t)$ é o vetor cujos elementos são os
resultados das derivadas de $\vec{r}$.

------------------------------------------------------------------------

Para o nosso problema, temos:

$$
\begin{cases}
\displaystyle\frac{dS}{dt} = - \beta S I \\[9pt]
\displaystyle\frac{dI}{dt} = \beta S I - \gamma I \\[9pt]
\displaystyle\frac{dR}{dt} = \gamma I
\end{cases}
$$

Transformando no modelo usado para o RK4, temos:

$$
\begin{aligned} 
\vec{r}(t) &= (S(t),\,I(t),\,R(t)) \\
\vec{f}(\vec{r}, t) &= (- \beta S I,\, \beta S I - \gamma I,\,\gamma I)
\end{aligned}
$$

e sendo $\vec{k}_i$ um vetor de 3 elementos, que seguem a mesma ordem de
$\vec{r}$.

## Comparando valores de $R_0$

Para iniciar a análise do modelo, será feita a comparação de três
gráficos, cada um com o valor de $R_0$ de $5$, $2.5$ e $2$. A
expectativa é que valores maiores do número básico de reprodução gerem
picos de infectados mais expressivos.

------------------------------------------------------------------------

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 200 # Para o Jupyter Notebook.

# Parâmetros.
N = 100_000

R0 = 0
I0 = 2
S0 = N - I0

gamma = 0.2

h = 0.1

In [2]:
def fSIR(y, t, beta):
  S, I, R = y

  dS = - beta * S * I
  dI = beta * S * I - gamma * I
  dR = gamma * I
  return np.array([dS, dI, dR])

def calc_beta(R_0):
  return R_0 * gamma / S0 # Obtido por: R_0 = (beta/gamma) * S(0)

def rk4_sys_SIR(f, ti, tf, h, y0, beta):
  tpoints = np.arange(ti, tf + h/2, h)   # +h/2 para ir acima do tf.
  n_steps = tpoints.size - 1

  # Este trecho inicializa a solução (com zeros).
  d = len(y0)
  ypoints = np.zeros((n_steps + 1, d))
  ypoints[0] = np.array(y0)

  y = ypoints[0].copy()
  for i in range(n_steps):
      t = tpoints[i]
      k1 = h * f(y, t, beta)
      k2 = h * f(y + 0.5 * k1, t + 0.5 * h, beta)
      k3 = h * f(y + 0.5 * k2, t + 0.5 * h, beta)
      k4 = h * f(y + k3, t + h, beta)
      y = y + (1.0/6.0) * (k1 + 2*k2 + 2*k3 + k4)
      ypoints[i+1] = y

  return tpoints, ypoints

def graficoSIR(R_0):
  y0 = [S0, I0, R0] # Condições iniciais.

  beta = calc_beta(R_0)

  t_sol, y_sol = rk4_sys_SIR(fSIR, 0, 120, h, y0, beta)

  y_sol = y_sol * 1e-4

  # Plot.
  plt.figure(figsize=(5, 3))
  plt.plot(t_sol, y_sol[:,0], 'g', label='Suscetíveis (S)')
  plt.plot(t_sol, y_sol[:,1], 'r', label='Infectados (I)')
  plt.plot(t_sol, y_sol[:,2], 'b', label='Removidos (R)')
  plt.xlabel('Tempo (dias)')
  plt.ylabel(r'Nº de indivíduos ($\times 10^4$)')
  plt.title('Simulação de epidemia')
  plt.legend()
  plt.show()

In [3]:
graficoSIR(5)

------------------------------------------------------------------------

In [4]:
graficoSIR(2.5)

------------------------------------------------------------------------

In [5]:
graficoSIR(2)

------------------------------------------------------------------------

Por fim, é interessante o fato de que valores menores de $R_0$ não
alcançam a contaminação de toda a população. Com isso, a conclusão que
tiramos é que, quanto menor for a taxa de contaminação de uma doença,
menor é seu alcance na população como um todo.

# Modelando o efeito da vacinação

## Vacinando apenas durante a epidemia

Até agora, vimos o comportamento da epidemia considerando que nada foi
feito para combatê-la.

Vamos considerar que uma vacina contra a doença foi desenvolvida, e que,
assim que a epidemia começou, inciou-se uma campanha de vacinação da
população.

Para isso, acrescentaremos um novo grupo ao modelo
$\textcolor{ForestGreen}{S}\textcolor{red}{I}\textcolor{blue}{R}$:

-   $\mathrm{\textcolor{purple}{Vacinados}}$: Foram vacinados contra a
    doença e ganharam imunidade permanente.

Com isto, teremos que:
$N = \textcolor{ForestGreen}{S}(t) + \textcolor{red}{I}(t) + \textcolor{blue}{R}(t) + \textcolor{purple}{V}(t)$.

## Modelo SIRV

$$
\frac{dS}{dt} = - \beta S I - \nu S \qquad \tag{6}
$$

$$
\frac{dV}{dt} = \nu S \tag{7}
$$

-   $\nu$: taxa de vacinação.

Esta modelagem da vacinação faz algumas suposições grosseiras:

-   Considera-se que a vacinação começou assim que os primeiros
    indivíduos foram infectados.
-   Considera-se que a taxa de vacinação é constante e proporcional ao
    número de suscetíveis.

## Comparando valores de $\nu$

Vamos comparar os gráficos de duas simulações com taxas de vacinação
$\nu$ iguais a $0.001$, $0.01$ e $0.05$, respectivamente, e com
$R_0 = 2.5$.

------------------------------------------------------------------------

In [6]:
# Parâmetros.
V0 = 0

R_0 = 2.5
beta = calc_beta(R_0)

In [7]:
def fSIRV(y, t, ni):
  S, I, R, V = y

  dS = - beta * S * I - ni * S
  dI = beta * S * I - gamma * I
  dR = gamma * I
  dV = ni * S
  return np.array([dS, dI, dR, dV])

def rk4_sys_SIRV(f, ti, tf, h, y0, ni):
  tpoints = np.arange(ti, tf + h/2, h)
  n_steps = tpoints.size - 1
  
  d = len(y0)
  ypoints = np.zeros((n_steps + 1, d))
  ypoints[0] = np.array(y0)

  y = ypoints[0].copy()
  for i in range(n_steps):
      t = tpoints[i]
      k1 = h * f(y, t, ni)
      k2 = h * f(y + 0.5 * k1, t + 0.5 * h, ni)
      k3 = h * f(y + 0.5 * k2, t + 0.5 * h, ni)
      k4 = h * f(y + k3, t + h, ni)
      y = y + (1.0/6.0) * (k1 + 2*k2 + 2*k3 + k4)
      ypoints[i+1] = y

  return tpoints, ypoints

def graficoSIRV(ni):
  y0 = [S0, I0, R0, V0]

  t_sol, y_sol = rk4_sys_SIRV(fSIRV, 0, 120, h, y0, ni)

  y_sol = y_sol * 1e-4

  plt.figure(figsize=(5, 3))
  plt.plot(t_sol, y_sol[:,0], 'g', label='Suscetíveis (S)')
  plt.plot(t_sol, y_sol[:,1], 'r', label='Infectados (I)')
  plt.plot(t_sol, y_sol[:,2], 'b', label='Removidos (R)')
  plt.plot(t_sol, y_sol[:,3], 'purple', label='Vacinados (V)')
  plt.xlabel('Tempo (dias)')
  plt.ylabel(r'Nº de indivíduos ($\times 10^4$)')
  plt.title('Simulação de epidemia com vacinação')
  plt.legend()
  plt.show()

In [8]:
graficoSIRV(0.001)

------------------------------------------------------------------------

In [9]:
graficoSIRV(0.01)

------------------------------------------------------------------------

In [10]:
graficoSIRV(0.05)

------------------------------------------------------------------------

In [11]:
def graficoSIRV_infectados(ni):
  y0 = [S0, I0, R0, V0]

  t_sol, y_sol = rk4_sys_SIRV(fSIRV, 0, 120, h, y0, ni)

  plt.figure(figsize=(5, 3))
  plt.plot(t_sol, y_sol[:,1], 'r', label=rf'$\nu = {ni}$')
  plt.xlabel('Tempo (dias)')
  plt.ylabel('Nº de Infectados (I)')
  plt.title('Simulação de epidemia com vacinação')
  plt.legend()
  plt.show()

In [12]:
graficoSIRV_infectados(0.05)

------------------------------------------------------------------------

Podemos concluir que:

-   Com uma taxa de vacinação suficientemente alta, grande parte da
    população não vacinada também não é infectada pela doença
    (imunidadade de rebanho).
-   A curva de infectados fica muito mais achatada. Isto poderia evitar
    um colapso do sistema de saúde (superlotação de hospitais, falta de
    medicamentos, entre outros problemas).

## Vacinando apenas antes da epidemia

Agora vamos considerar uma outra situação: uma parte da população foi
vacinada antes da epidemia começar, e não houve campanha de vacinação
durante a epidemia.

Neste caso, o número de vacinados $V$ se torna constante:
$V = \rho S(0)$. Logo, o número de indivíduos suscetíveis no início da
epidemia passa a ser $S(0) - V$, ou $(1 - \rho) S(0)$ (desconsiderando
os infectados).

------------------------------------------------------------------------

Pensando no oposto da equação $(4)$, a proporção mínima da população que
deve estar vacinada para que uma epidemia não ocorra (imunidade de
rebanho) é:

$$
\frac{\beta}{\gamma} (1 - \rho) S(0) \le 1
$$

$$
(1 - \rho) R_0 \le 1 \qquad (R_0 = (\beta / \gamma) S(0))
$$

$$
\rho \ge 1 - \frac{1}{R_0} \tag{8}
$$

## Comparando valores de $\rho$

Vamos fazer três simulações de epidemia, com $\rho$ valendo $0.3$, $0.5$
e $0.6$, respectivamente, e com o número inicial de infectados
$I(0) = 100$.

------------------------------------------------------------------------

In [13]:
# Parâmetros.
I0 = 100
S0_normal = N - I0

In [14]:
ro = 0.3
V0 = ro * S0_normal
S0 = S0_normal - V0

graficoSIRV(0)

------------------------------------------------------------------------

In [15]:
ro = 0.5
V0 = ro * S0_normal
S0 = S0_normal - V0

graficoSIRV(0)

------------------------------------------------------------------------

In [16]:
ro = 0.6
V0 = ro * S0_normal
S0 = S0_normal - V0

graficoSIRV(0)

------------------------------------------------------------------------

In [17]:
def graficoSIRV_infectados_ro():
  ro = 0.5
  V0 = ro * S0_normal
  S0 = S0_normal - V0

  y0 = [S0, I0, R0, V0]

  t1_sol, y1_sol = rk4_sys_SIRV(fSIRV, 0, 120, h, y0, 0)

  ro = 0.6
  V0 = ro * S0_normal
  S0 = S0_normal - V0

  y0 = [S0, I0, R0, V0]

  t2_sol, y2_sol = rk4_sys_SIRV(fSIRV, 0, 120, h, y0, 0)

  plt.figure(figsize=(5, 3))
  plt.plot(t1_sol, y1_sol[:,1], 'r--', label=rf'$\rho = 0.5$')
  plt.plot(t2_sol, y2_sol[:,1], 'r', label=rf'$\rho = 0.6$')
  plt.xlabel('Tempo (dias)')
  plt.ylabel('Nº de Infectados (I)')
  plt.title('Simulação de epidemia com vacinação')
  plt.legend()
  plt.show()

In [18]:
graficoSIRV_infectados_ro()

------------------------------------------------------------------------

Podemos ver que, quando maior é a proporção da população que já está
vacinada, mais achatada se torna a curva de infectados.

## Bônus

Vamos pensar num caso mais completo de epidemia.

-   A população é bem maior: 5 milhões de habitantes.
-   Inicialmente, uma única pessoa se infecta com a doença.
-   A taxa de transmissão é de 3 pessoas por dia, e a taxa de
    recuperação é de 0.1.
-   Já existia vacina para a doença antes da epidemia, mas apenas 30% da
    população estava vacinada.
-   Assim que o surto começou, começou-se uma campanha de vacinação, a
    uma taxa de 0.01% da população saudável por dia.

------------------------------------------------------------------------

In [19]:
# Parâmetros.
N = 5_000_000

I0 = 1
S0_normal = N - I0
ro = 0.3
V0 = ro * S0_normal
S0 = S0_normal - V0

R_0 = 3
beta = calc_beta(R_0)
gamma = 0.1
ni = 0.01

In [20]:
graficoSIRV(ni)